# Gesture Controller — Notebook port of `gesture_controller0.py`

This notebook is adapted for environments matching **Python 3.6.9** with pinned package versions. It contains compatibility shims, dependency checks, unit tests and a notebook-friendly realtime pipeline to control a local media player (VLC) via hand gestures using MediaPipe.

Sections:
1. Environment & version pinning
2. Install / verify dependencies (pinned)
3. Model download & integrity check
4. MediaPipe initialization (Tasks API + legacy fallback)
5. Helper functions (drawing, finger count)
6. Gesture mapping & stabiliser (tests included)
7. VLC / keyboard dispatcher (xdotool on Linux, pynput fallback)
8. Main realtime loop (camera → detect → dispatch)
9. Performance tracker & analysis
10. Unit tests
11. Launch cell & troubleshooting

Note: cells are split and safe to run in order; the live loop cell will open the camera when executed.

In [ ]:
# Section 1 — Environment & version pinning (report-only)
import sys
import platform
import importlib

# Target (pinned) versions suitable for Python 3.6.9 environments
TARGET_PYTHON = '3.6.9'
TARGET_NUMPY = '1.19.5'
TARGET_OPENCV = '4.5.5.64'
TARGET_MEDIAPIPE = '0.8.10'
TARGET_PYNPUT = '1.6.8'
TARGET_MATPLOTLIB = '3.3.4'

print('Runtime Python :', sys.version.split()[0], '(', platform.python_implementation(), ')')
print('Target Python  :', TARGET_PYTHON)

def _ver(pkg):
    try:
        m = importlib.import_module(pkg)
        return getattr(m, '__version__', 'unknown')
    except Exception:
        return 'not installed'

print('numpy    :', _ver('numpy'))
print('cv2      :', _ver('cv2'))
print('mediapipe:', _ver('mediapipe'))
print('pynput   :', _ver('pynput'))
print('matplotlib:', _ver('matplotlib'))

# Simple check — only *report* (no enforcement)
if sys.version.split()[0] != TARGET_PYTHON:
    print('\n⚠️  Python version mismatch — notebook targets Python', TARGET_PYTHON)
    print('    (This notebook prints warnings for versions; it will still attempt to run.)')
else:
    print('\n✓ Python version matches target')


## Section 2 — Install / verify dependencies (pinned)

The next cell shows `pip` commands pinned to versions compatible with Python 3.6.9. Run them only if you need to install or reproduce the pinned environment.

In [ ]:
# PIP commands (copy/paste to your shell if you need to install)
print('pip install commands for pinned environment:')
print('\n'.join([
    f"pip install numpy=={TARGET_NUMPY}",
    f"pip install opencv-python=={TARGET_OPENCV}",
    f"pip install mediapipe=={TARGET_MEDIAPIPE}",
    f"pip install pynput=={TARGET_PYNPUT}",
    f"pip install matplotlib=={TARGET_MATPLOTLIB}",
]))

# Lightweight import checks (do not auto-install in the notebook)
def check_imports():
    missing = []
    for pkg in ('numpy','cv2','mediapipe','pynput','matplotlib'):
        try:
            __import__(pkg)
        except Exception:
            missing.append(pkg)
    if missing:
        print('Missing packages:', missing)
        print('Install the pinned packages shown above in a shell.')
    else:
        print('All pinned packages appear importable (in current environment).')

check_imports()


## Section 3 — Model file download & local path check

Ensures `hand_landmarker.task` exists in the project folder and downloads it if missing. Performs a simple size/checksum verification.

In [ ]:
import os
import hashlib

MODEL_FILENAME = 'hand_landmarker.task'
MODEL_PATH = os.path.join(os.getcwd(), MODEL_FILENAME)
EXPECTED_MIN_SIZE = 2000000   # bytes — sanity check (example)

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

if os.path.isfile(MODEL_PATH):
    size = os.path.getsize(MODEL_PATH)
    print('Model file found:', MODEL_PATH)
    print('Size (bytes):', size)
    if size < EXPECTED_MIN_SIZE:
        print('⚠️  Model file smaller than expected — it may be corrupted.')
    else:
        try:
            print('SHA256:', _sha256(MODEL_PATH)[:16], '...')
        except Exception:
            print('Could not compute checksum (permission error).')
else:
    print('Model not found locally. To download:')
    print('  curl -o', MODEL_FILENAME, 'https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/hand_landmarker.task')
    print('After download, re-run this cell to verify the model file.')


## Section 4 — MediaPipe initialization (Tasks API + legacy fallback)

Creates a `hand_detector` object usable by downstream cells. The code provides shims so the result format (Tasks API vs legacy solutions) can be handled uniformly.

In [ ]:
import mediapipe as mp

USE_TASKS = False
hand_detector = None

try:
    # Try Tasks API (preferred when model file available)
    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision as mp_vision

    if os.path.isfile(MODEL_PATH):
        base_opts = mp_tasks.BaseOptions(model_asset_path=MODEL_PATH)
        hand_opts = mp_vision.HandLandmarkerOptions(
            base_options=base_opts,
            num_hands=1,
            min_hand_detection_confidence=0.5,
            min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            running_mode=mp_vision.RunningMode.VIDEO,
        )
        hand_detector = mp_vision.HandLandmarker.create_from_options(hand_opts)
        USE_TASKS = True
        print('✓ MediaPipe Tasks API initialized (model used)')
    else:
        raise RuntimeError('Model file missing — fallback to legacy API')
except Exception as e:
    print('Tasks API unavailable or model missing:', e)
    print('Falling back to mediapipe.solutions.hands')
    mp_hands = mp.solutions.hands
    hand_detector = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                                   min_detection_confidence=0.5, min_tracking_confidence=0.5)

mp_drawing = mp.solutions.drawing_utils
mp_connections = mp.solutions.hands.HAND_CONNECTIONS
print('USE_TASKS =', USE_TASKS)


## Section 5 — Helper functions (drawing, finger counting)

Pure functions are unit-testable and compatible with Python 3.6.9.

In [ ]:
def draw_landmarks(img, hand_landmarks):
    """Draw simple lines/circles for a list of normalized landmarks (0..1).
    Accepts either a Tasks-landmark list or legacy LandmarkList.
    """
    if not hand_landmarks:
        return
    h, w = img.shape[:2]
    # hand_landmarks may be a protobuf list or a plain list of objects
    for lm in hand_landmarks:
        x, y = int(lm.x * w), int(lm.y * h)
        cv2.circle(img, (x, y), 3, (255, 0, 0), -1)


def count_fingers_from_landmarks(landmarks, is_right=True):
    """Return list [T, I, M, R, P] with 0/1 per finger.
    landmarks: list-like of points with .x and .y
    """
    tips = [4, 8, 12, 16, 20]
    fingers = []
    # Thumb: compare tip vs IP on x-axis (simple heuristic)
    try:
        if is_right:
            fingers.append(1 if landmarks[4].x < landmarks[3].x else 0)
        else:
            fingers.append(1 if landmarks[4].x > landmarks[3].x else 0)
    except Exception:
        fingers.append(0)
    # Other fingers: tip higher (y smaller) than pip (i-2)
    for t in tips[1:]:
        try:
            fingers.append(1 if landmarks[t].y < landmarks[t-2].y else 0)
        except Exception:
            fingers.append(0)
    return fingers

# Quick pure-function unit check
assert count_fingers_from_landmarks.__name__ == 'count_fingers_from_landmarks'
print('✓ Helper functions defined (draw_landmarks, count_fingers_from_landmarks)')

## Section 6 — Gesture map, stabiliser & cooldown logic

Includes deterministic tests for the stabiliser behaviour.

In [ ]:
GESTURE_PATTERNS = {
    (1,1,1,1,1): ('play_pause', 'Play / Pause'),
    (0,0,0,0,0): ('mute', 'Mute / Unmute'),
    (1,0,0,0,0): ('vol_up', 'Volume Up'),
    (0,0,0,0,1): ('vol_down', 'Volume Down'),
    (0,1,1,0,0): ('next_track', 'Next Track'),
    (0,1,0,0,0): ('prev_track', 'Previous Track'),
    (0,1,1,1,0): ('fullscreen', 'Fullscreen Toggle'),
}

class GestureStabiliser(object):
    def __init__(self, hold=3, cooldown=1.0):
        self.hold = hold
        self.cooldown = cooldown
        self._prev = None
        self._streak = 0
        self._last = {}

    def update(self, pattern):
        # Accept None to reset
        if pattern is None:
            self._prev = None
            self._streak = 0
            return None
        if pattern == self._prev:
            self._streak += 1
        else:
            self._prev = pattern
            self._streak = 1

        if self._streak < self.hold:
            return None

        act = GESTURE_PATTERNS.get(pattern)
        if not act:
            return None
        name = act[0]
        import time as _t
        now = _t.time()
        if now - self._last.get(name, 0) < self.cooldown:
            return None
        self._last[name] = now
        return act

# Deterministic test for the stabiliser
s = GestureStabiliser(hold=2, cooldown=0.1)
assert s.update((1,0,0,0,0)) is None
assert s.update((1,0,0,0,0)) is not None
print('✓ GestureStabiliser basic test passed')


## Section 7 — VLC / keyboard dispatcher (xdotool on Jetson, pynput otherwise)

Supports a `dry_run` mode for CI/testing that logs actions instead of sending keys.

In [ ]:
import subprocess
import platform

def _dry_log(action_name):
    print('[dry-run] action ->', action_name)

IS_LINUX = platform.system() == 'Linux'

def dispatch_action(action_name, dry_run=False):
    """Dispatch the media-control action. Returns True if dispatched or logged."""
    if dry_run:
        _dry_log(action_name)
        return True

    if IS_LINUX:
        xmap = {
            'play_pause': 'space', 'mute': 'm',
            'vol_up': 'ctrl+Up', 'vol_down': 'ctrl+Down',
            'next_track': 'n', 'prev_track': 'p', 'fullscreen': 'f'
        }
        k = xmap.get(action_name)
        if not k:
            return False
        try:
            subprocess.Popen(['xdotool', 'key', k], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return True
        except Exception:
            print('xdotool not available or failed to send keys')
            return False
    else:
        try:
            from pynput.keyboard import Controller, Key
            kb = Controller()
            if action_name == 'play_pause':
                kb.press(' '); kb.release(' ')
            elif action_name == 'mute':
                kb.press('m'); kb.release('m')
            elif action_name == 'vol_up':
                kb.press(Key.up); kb.release(Key.up)
            elif action_name == 'vol_down':
                kb.press(Key.down); kb.release(Key.down)
            elif action_name == 'next_track':
                kb.press('n'); kb.release('n')
            elif action_name == 'prev_track':
                kb.press('p'); kb.release('p')
            elif action_name == 'fullscreen':
                kb.press('f'); kb.release('f')
            else:
                return False
            return True
        except Exception:
            print('pynput not available or failed to send keys')
            return False

print('✓ Dispatcher ready (dry-run supported)')


## Section 8 — Main realtime loop (camera, detect, classify, dispatch, OSD)

Call `run_gesture_control()` to start the live loop. The function accepts `dry_run` and `low_res` flags for safe testing and Jetson performance mode.

In [ ]:
def run_gesture_control(camera_src=0, low_res=False, dry_run=False, max_frames=None):
    import time
    cap = cv2.VideoCapture(camera_src)
    if low_res:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    if not cap.isOpened():
        print('ERROR: cannot open camera')
        return

    print('Camera opened. Focus VLC window if you want keyboard events to reach it.')

    frame_count = 0
    last_action_time = 0
    cooldown = 1.0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        landmarks_list = None
        handedness_label = 'Right'

        # Detection (Tasks API vs legacy)
        if USE_TASKS:
            try:
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                result = hand_detector.detect_for_video(mp_image, int(time.time()*1000))
                if result.hand_landmarks:
                    landmarks_list = result.hand_landmarks[0]
                    handedness_label = result.handedness[0][0].category_name
            except Exception:
                landmarks_list = None
        else:
            results = hand_detector.process(rgb)
            if results.multi_hand_landmarks and results.multi_handedness:
                landmarks_list = results.multi_hand_landmarks[0]
                handedness_label = results.multi_handedness[0].classification[0].label

        pattern = None
        if landmarks_list:
            draw_landmarks(frame, landmarks_list)
            pattern = tuple(count_fingers_from_landmarks(landmarks_list, is_right=(handedness_label=='Right')))

        # classify & dispatch
        if pattern:
            act = GESTURE_PATTERNS.get(pattern)
            if act:
                now = time.time.time() if False else time.time()
                if now - last_action_time > cooldown:
                    dispatched = dispatch_action(act[0], dry_run=dry_run)
                    if dispatched:
                        print('Action ->', act[1])
                        last_action_time = now

        # OSD
        cv2.putText(frame, 'Pattern: {}'.format(pattern), (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        cv2.imshow('Gesture Control', frame)

        key = cv2.waitKey(1) & 0xFF
        if key in (27, ord('q')):
            break
        frame_count += 1
        if max_frames and frame_count >= max_frames:
            break

    cap.release()
    cv2.destroyAllWindows()
    print('Run finished')

print('✓ run_gesture_control defined (call with dry_run=True for safe tests)')

## Section 9 — Performance tracker & post-run analysis

Lightweight tracker to capture FPS and latencies (use after running the live loop).

In [ ]:
class SimplePerf:
    def __init__(self):
        self.times = []
    def push(self, t):
        self.times.append(t)
    def summary(self):
        if not self.times:
            return 'No samples'
        import numpy as _np
        arr = _np.array(self.times)
        return 'frames=%d  mean=%.1fms  p95=%.1fms' % (len(arr), arr.mean(), _np.percentile(arr,95))

perf = SimplePerf()
print('✓ Simple performance tracker created')


## Section 10 — Unit tests for helpers & stabiliser

Small, pytest-style checks that can be run inside the notebook (assert-based for simplicity).

In [ ]:
# Basic in-notebook tests (assert-based)
# 1) fingers logic (synthetic landmarks)
class _P: pass
lm = [ _P() for _ in range(21) ]
# create synthetic coordinates where index is up and others down
for i in range(21):
    setattr(lm[i], 'x', 0.5)
    setattr(lm[i], 'y', 0.9)
# make index tip higher than pip
lm[8].y = 0.2
lm[6].y = 0.6
res = count_fingers_from_landmarks(lm, is_right=True)
assert res[1] == 1 and sum(res) == 1

# 2) stabiliser replay
s = GestureStabiliser(hold=2, cooldown=0.1)
assert s.update((0,1,0,0,0)) is None
assert s.update((0,1,0,0,0)) is not None

print('✓ In-notebook unit tests passed')


## Section 11 — Launch cell & run instructions

To run the live demo: 1) ensure VLC is open and focused; 2) run the Environment & Model-check cells; 3) run the `run_gesture_control()` cell below (use `dry_run=True` first to test).

In [ ]:
# ▶ Launch (example) — uncomment to run
# run_gesture_control(camera_src=0, low_res=True, dry_run=True, max_frames=200)

print('Ready — run the commented call above to start a dry-run demo')

## Section 12 — Troubleshooting & diagnostics

- Camera enumeration helper
- Model integrity test
- xdotool / pynput checks

Run these if you hit permission or device errors.

In [ ]:
# Diagnostics helpers
import subprocess

def list_cameras(max_id=6):
    found = []
    import cv2 as _cv
    for i in range(max_id):
        c = _cv.VideoCapture(i)
        ok, _ = c.read()
        c.release()
        if ok:
            found.append(i)
    return found

print('Available camera indices (quick scan):', list_cameras())

# xdotool check (Linux)
import platform as _plat
if _plat.system() == 'Linux':
    try:
        subprocess.check_output(['xdotool', '--version'])
        print('xdotool: available')
    except Exception:
        print('xdotool: NOT available — install `sudo apt install xdotool` to enable keyboard dispatch')
else:
    print('Non-Linux OS detected; pynput will be used for keyboard dispatch')

# Model quick-load test (non-invasive)
if os.path.isfile(MODEL_PATH):
    print('Model file present — size bytes =', os.path.getsize(MODEL_PATH))
else:
    print('Model file missing — see Section 3 for download instructions')
